### Imports and Setup

In [3]:
import anthropic
import os
import json
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv("../backend/.env")

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
engine = create_engine(f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}")

### Definition of categories and prompt

In [9]:
CATEGORIES = [
    "Income", "Groceries", "Dining", "Coffee", 
    "Transportation", "Shopping", "Subscriptions", 
    "Health & Wellness", "Transfer", "Other"
]

system_prompt = f"""You are a financial transaction categorizer for a Canadian personal finance app.

You will receive a bank transaction description and an optional transaction code.
Transaction codes mean:
- DN: direct deposit (likely Income or Transfer)
- CW: cash withdrawal or e-transfer sent (likely Transfer)
- PR: point of sale purchase (likely Shopping, Groceries, Dining, Coffee etc.)
- OP: online purchase (likely Shopping, Subscriptions etc.)

Respond ONLY with a JSON object in this exact format: {{"category": "CategoryName"}}
Choose from this exact list: {CATEGORIES}

Rules:
- Return only valid JSON, no explanation, no markdown, no extra text
- If uncertain, return "Other" — never return null
- Canadian context: CRA deposits are Income, Compass card is Transportation

Canadian specific rules:
- TLNK or TRANSLINK or COMPASS VENDING = Transportation
- CRA or CANADA REVENUE = Income  
- BC REVENUE or BC SERVICES = Health & Wellness
- TF followed by numbers = Transfer

Example:
Input: [PR] TIM HORTONS VANCOUVER BC
Output: {{"category": "Coffee"}}
"""

### Function to categorize one transaction

In [10]:
def categorize_transaction(description, transaction_code):
    """
    sends a transaction description to Calude API.
    Returns a valid category from categories list. 
    """
    
    user_message = f"Transaction code: {transaction_code}\nDescription: {description}"
    
    
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=50,
        system=system_prompt,
        messages=[{
            "role":"user",
            "content": user_message,
        }]
    )
    
    raw = response.content[0].text.strip()
    
    try:
        parsed = json.loads(raw)
        category = parsed.get("category", "Other")
        if category not in CATEGORIES:
            return "Other"
        return category
    except json.JSONDecodeError:
        return "Other"
        

### Fetch Null rows and run the loop

In [7]:
with engine.connect() as connection:
    result = connection.execute(text("""
    SELECT id, description, transaction_code
    FROM transactions
    WHERE category IS NULL
    LIMIT 5
    """))
    
    rows = result.fetchall()
    print(f"Testing with {len(rows)} rows.")
    
with engine.connect() as connection:
    for row in rows:
        print(f"description: '{row.description}', code: '{row.transaction_code}'")
        category = categorize_transaction(row.description, row.transaction_code)
        connection.execute(text("""
        UPDATE transactions
        SET category = :category
        WHERE id = :id
        """), {"category":category, "id":row.id})
        
        print(f"{row.description[:40]} → {category}")
        
    connection.commit()
    print("Done.")

In [8]:
with engine.connect() as connection:
    # fix TLNK → Transportation
    connection.execute(text("""
        UPDATE transactions
        SET category = 'Transportation'
        WHERE id = 3
    """))
    
    # fix BC Revenue Services → you decide the category
    connection.execute(text("""
        UPDATE transactions
        SET category = 'Health & Wellness'
        WHERE id = 4
    """))
    
    connection.commit()
    print("Fixed manually")